## Work Done By: Mohammad Al-Refaie

## 1- Imports

In [ ]:
import os
import re
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm
import PyPDF2
import docx
from pptx import Presentation

In [ ]:
device = torch.device("cpu")

## 2- File Type Handlers

In [34]:
def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    if ext == '.txt':
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
    elif ext == '.pdf':
        with open(file_path, 'rb') as f:
            pdf = PyPDF2.PdfReader(f)
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    elif ext == '.docx':
        doc = docx.Document(file_path)
        for para in doc.paragraphs:
            if para.text:
                text += para.text + "\n"
    elif ext == '.pptx':
        prs = Presentation(file_path)
        for slide in prs.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text:
                    text += shape.text + "\n"
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    return text

## 3- Text Cleaning and Chunking

In [35]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+', '', text)
    text = re.sub(r'[^\w\s\.\,\!\?]', '', text)
    return text.strip()

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.split()) >= 5]

def chunk_sentences(sentences, max_tokens=300):
    chunks = []
    current_chunk = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(sent.split())
        if current_tokens + sent_tokens > max_tokens and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sent]
            current_tokens = sent_tokens
        else:
            current_chunk.append(sent)
            current_tokens += sent_tokens
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

## 4- Model Loading

In [ ]:
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name, use_safetensors=True)
model.to(device)
model.eval()

## 5- Question Generation with Difficulty Control

In [ ]:
def generate_questions(chunk, difficulty, num_per_chunk=2):
    """
    Generate multiple MCQs from a chunk. Retries if parsing fails.
    """
    all_questions = []
    max_attempts = 3

    for _ in range(num_per_chunk):
        for attempt in range(max_attempts):
            # Build prompt
            if difficulty == "easy":
                instruction = "Create an easy multiple-choice question with 4 options and the correct answer. Output each part on its own line:\nQuestion: ...\nA) ...\nB) ...\nC) ...\nD) ...\nAnswer: [A-D]"
            elif difficulty == "hard":
                instruction = "Create a difficult multiple-choice question that requires reasoning. Output each part on its own line:\nQuestion: ...\nA) ...\nB) ...\nC) ...\nD) ...\nAnswer: [A-D]"
            else:
                instruction = "Create a multiple-choice question with 4 options and the correct answer. Output each part on its own line:\nQuestion: ...\nA) ...\nB) ...\nC) ...\nD) ...\nAnswer: [A-D]"

            prompt = f"{instruction}\n\nContext:\n{chunk}"
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

            if difficulty == "hard":
                outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.8, do_sample=True, top_p=0.9)
            else:
                outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False)

            generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Parse MCQ
            lines = generated.split('\n')
            question = None
            options = [None, None, None, None]
            answer = None
            for line in lines:
                line = line.strip()
                if line.startswith("Question:") or line.startswith("Q:"):
                    question = line.split(":", 1)[1].strip()
                elif line.startswith("A)") or line.startswith("A."):
                    options[0] = line[2:].strip()
                elif line.startswith("B)") or line.startswith("B."):
                    options[1] = line[2:].strip()
                elif line.startswith("C)") or line.startswith("C."):
                    options[2] = line[2:].strip()
                elif line.startswith("D)") or line.startswith("D."):
                    options[3] = line[2:].strip()
                elif line.startswith("Answer:") or line.startswith("Correct answer:"):
                    ans_letter = line.split(":", 1)[1].strip().upper()
                    if ans_letter in ['A','B','C','D']:
                        answer = ans_letter

            # If not found, try regex for concatenated format
            if not question or any(o is None for o in options) or not answer:
                # Extract options using a more flexible pattern
                opts = re.findall(r'[A-D][\)\.]\s*(.*?)(?=\s*[A-D][\)\.]|\s*Answer:)', generated, re.DOTALL)
                if len(opts) >= 4:
                    options = [o.strip() for o in opts[:4]]
                # Extract question
                q_match = re.search(r"Question:\s*(.*?)(?=\s*Options?:|\s*[A-D][\)\.])", generated, re.DOTALL | re.IGNORECASE)
                if q_match:
                    question = q_match.group(1).strip()
                # Extract answer
                ans_match = re.search(r"Answer:\s*([A-D])", generated, re.IGNORECASE)
                if ans_match:
                    answer = ans_match.group(1).upper()

            # If we have everything, add to list
            if question and all(o is not None for o in options) and answer:
                answer_text = options[ord(answer) - ord('A')]
                all_questions.append({
                    "type": "mcq",
                    "question": question,
                    "options": options,
                    "correct_answer": answer_text,
                })
                break  # success, move to next question
            # else, try again (up to max_attempts)
        else:
            # Could not generate after max_attempts, skip
            pass

    return all_questions    

In [59]:
def generate_exam(file_path, difficulty, num_questions=10):
    raw_text = extract_text(file_path)
    if not raw_text:
        return "No text extracted from file."

    cleaned = clean_text(raw_text)
    sentences = split_into_sentences(cleaned)
    chunks = chunk_sentences(sentences, max_tokens=300)

    questions = []
    num_chunks = len(chunks)
    if num_chunks == 0:
        return "No chunks generated."

    # Generate a little extra to cover failures
    per_chunk = max(1, (num_questions + num_chunks - 1) // num_chunks) + 1

    for chunk in tqdm(chunks, desc="Generating questions"):
        if len(questions) >= num_questions:
            break
        new_qs = generate_questions(chunk, difficulty, num_per_chunk=per_chunk)
        questions.extend(new_qs)

    # Trim to exactly the requested number
    questions = questions[:num_questions]

    # Format output
    output = f"# Exam generated from {os.path.basename(file_path)}\n"
    output += f"Difficulty: {difficulty}  |  Type: Multiple Choice\n\n"

    for i, q in enumerate(questions, 1):
        output += f"{i}. {q['question']}\n"
        for opt_letter, opt_text in zip(['A', 'B', 'C', 'D'], q['options']):
            if opt_text:
                output += f"   {opt_letter}. {opt_text}\n"
        if q.get('correct_answer'):
            output += f"   **Answer:** {q['correct_answer']}\n"
        output += "\n"

    return output

## 6- Example Usage

In [ ]:
if __name__ == "__main__":
    file_path = "Your File Path"
    if (file_path.startswith('"') and file_path.endswith('"')) or (file_path.startswith("'") and file_path.endswith("'")):
        file_path = file_path[1:-1]  # strip quotes

    if not os.path.exists(file_path):
        print("File not found. Please check the path.")
        folder = os.path.dirname(file_path)
        if os.path.exists(folder):
            print(f"\nContents of '{folder}':")
            for f in os.listdir(folder):
                print(f"  {f}")
        exit()
    else:
        print("File found!")

    # Ask for difficulty
    difficulty = input("Choose difficulty (easy / medium / hard): ").strip().lower()
    while difficulty not in ["easy", "medium", "hard"]:
        difficulty = input("Invalid. Please choose easy, medium, or hard: ").strip().lower()

    # Ask for number of questions
    try:
        num_q = int(input("How many questions? (default 10): ") or "10")
    except:
        num_q = 10

    # Generate exam
    exam_text = generate_exam(file_path, difficulty, num_q)
    print("\n" + "="*50)
    print(exam_text)
    print("="*50)

    # Save to file
    output_file = "generated_exam.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(exam_text)
    print(f"\nExam saved to {output_file}")